In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

In [18]:
# ============================================================
# 1. KONFIGURASI
# ============================================================

TRAIN_CSV_PATH = r"D:\KULIAH\RISET\PROGRAM\RESULT\SHOPPERS\MCAR\60\CSV\train_impute_mrmd_0.csv"
TEST_CSV_PATH  = r"D:\KULIAH\RISET\PROGRAM\RESULT\SHOPPERS\MCAR\60\CSV\test_impute_mrmd_0.csv"
INFO_JSON_PATH = r"D:\KULIAH\RISET\PROGRAM\RESULT\datasets\info\shoppers.json"

RANDOM_STATE = 42
N_SPLITS_CV = 5


In [33]:
# ============================================================
# 1. KONFIGURASI UNTUK MEAN MODUS
# ============================================================

BASE_DIR = r"D:\KULIAH\RISET\PROGRAM\RESULT\baselines\imputed_csv"

DATASET_NAME = "adult"
MASK = "mask_0"

TRAIN_CSV_PATH = rf"{BASE_DIR}\{DATASET_NAME}\{MASK}\mean_mode_train.csv"
TEST_CSV_PATH  = rf"{BASE_DIR}\{DATASET_NAME}\{MASK}\mean_mode_test.csv"

INFO_JSON_PATH = r"D:\KULIAH\RISET\PROGRAM\RESULT\datasetS\info\adult.json"

RANDOM_STATE = 42
N_SPLITS_CV = 5

In [34]:
# ============================================================
# 2. AMBIL NAMA KOLOM TARGET DARI info.json
# ============================================================

with open(INFO_JSON_PATH, "r") as f:
    info = json.load(f)

if "target_col" in info:
    target_col = info["target_col"]
elif "target_col_idx" in info:
    idx_list = info["target_col_idx"]
    idx = idx_list[0] if isinstance(idx_list, list) else idx_list
    target_col = info["column_names"][idx] if "column_names" in info else idx
else:
    raise ValueError(f"Tidak ada 'target_col'/'target_col_idx' di info.json. Key: {list(info.keys())}")

print(f"Target column: {target_col}")

Target column: 14


In [32]:
# ============================================================
# 3. LOAD DATA (train & test sudah terpisah, tidak displit lagi)
# ============================================================

df_train = pd.read_csv(TRAIN_CSV_PATH)
df_test = pd.read_csv(TEST_CSV_PATH)

if isinstance(target_col, int):
    target_col = df_train.columns[target_col]

y_train = df_train[target_col]
X_train = df_train.drop(columns=[target_col])

y_test = df_test[target_col]
X_test = df_test.drop(columns=[target_col])

print(f"Shape X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Shape X_test : {X_test.shape}, y_test : {y_test.shape}")
print(f"Distribusi kelas (train):\n{y_train.value_counts()}")
print(f"Distribusi kelas (test):\n{y_test.value_counts()}")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\KULIAH\\RISET\\PROGRAM\\RESULT\\baselines\\hyperimpute\\stat_baseline_output\\MCAR\\imputed_csv\\adult\\mask_0\\mean_mode_train.csv'

In [35]:
# ============================================================
# 3. LOAD DATA MEAN MODUS
# ============================================================

df_train = pd.read_csv(TRAIN_CSV_PATH)
df_test = pd.read_csv(TEST_CSV_PATH)

if isinstance(target_col, int):
    target_col = df_train.columns[target_col]

y_train = df_train[target_col]
X_train = df_train.drop(columns=[target_col])

y_test = df_test[target_col]
X_test = df_test.drop(columns=[target_col])

print(f"Shape X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Shape X_test : {X_test.shape}, y_test : {y_test.shape}")
print(f"Distribusi kelas (train):\n{y_train.value_counts()}")
print(f"Distribusi kelas (test):\n{y_test.value_counts()}")

Shape X_train: (22792, 14), y_train: (22792,)
Shape X_test : (9769, 14), y_test : (9769,)
Distribusi kelas (train):
14
<=50K    17273
>50K      5519
Name: count, dtype: int64
Distribusi kelas (test):
14
<=50K    7447
>50K     2322
Name: count, dtype: int64


In [36]:
# ============================================================
# 4. ENCODING
# ============================================================

# encode kolom kategorikal di fitur — fit di train, transform di train & test
for col in X_train.select_dtypes(include=["object", "category"]).columns:
    le = LabelEncoder()
    le.fit(X_train[col].astype(str))

    # tangani label baru di test yang mungkin tidak muncul di train
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = X_test[col].astype(str).map(
        lambda v: le.transform([v])[0] if v in le.classes_ else -1
    )

# encode target — fit di train, transform di train & test
y_train = y_train.astype(str).str.strip()
y_test = y_test.astype(str).str.strip()

le_target = LabelEncoder()
le_target.fit(y_train)
y_train = le_target.transform(y_train)
y_test = le_target.transform(y_test)

C:\Users\RETNO\AppData\Local\Temp\ipykernel_25340\3699339873.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X_train.select_dtypes(include=["object", "category"]).columns:


In [37]:
# ============================================================
# 6. DEFINISI MODEL (HYPERPARAMETER DEFAULT)
# ============================================================

classifiers = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(random_state=RANDOM_STATE)),  # default hyperparameter
    ]),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(probability=True, random_state=RANDOM_STATE)),  # default hyperparameter
    ]),
    "XGBoost": XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
    ),  # default hyperparameter
}

# BASELINE

In [15]:
# ============================================================
# 7. TRAINING + CROSS VALIDATION (dari data TRAINING)
# ============================================================

cv = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_weighted",
    "roc_auc": "roc_auc",
}

cv_results = {}
test_results = {}   # <-- tambahan: simpan hasil evaluasi test set

for name, clf in classifiers.items():
    print(f"\n{'='*50}\nModel: {name}\n{'='*50}")

    # ---- Cross-validation di data training ----
    scores = cross_validate(clf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_results[name] = {
        metric: (np.mean(scores[f"test_{metric}"]), np.std(scores[f"test_{metric}"]))
        for metric in scoring
    }

    print("--- Cross-Validation (Training Set) ---")
    for metric, (mean_val, std_val) in cv_results[name].items():
        print(f"{metric:10s}: {mean_val:.4f} (+/- {std_val:.4f})")

    # ---- Fit ke seluruh training set, evaluasi ke test set ----
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]  # asumsi biner

    test_metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted"),
        "recall": recall_score(y_test, y_pred, average="weighted"),
        "f1": f1_score(y_test, y_pred, average="weighted"),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
    test_results[name] = test_metrics   # <-- simpan untuk ringkasan nanti

    print("\n--- Evaluasi di Test Set ---")
    for metric, val in test_metrics.items():
        print(f"{metric:10s}: {val:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


Model: LogisticRegression
--- Cross-Validation (Training Set) ---
accuracy  : 0.8563 (+/- 0.0026)
precision : 0.8346 (+/- 0.0073)
recall    : 0.8563 (+/- 0.0026)
f1        : 0.8144 (+/- 0.0055)
roc_auc   : 0.7647 (+/- 0.0109)

--- Evaluasi di Test Set ---
accuracy  : 0.8610
precision : 0.8425
recall    : 0.8610
f1        : 0.8202
roc_auc   : 0.7662

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.99      0.92      3135
           1       0.71      0.15      0.25       564

    accuracy                           0.86      3699
   macro avg       0.79      0.57      0.58      3699
weighted avg       0.84      0.86      0.82      3699

Confusion Matrix:
[[3101   34]
 [ 480   84]]

Model: SVM
--- Cross-Validation (Training Set) ---
accuracy  : 0.8536 (+/- 0.0027)
precision : 0.8354 (+/- 0.0110)
recall    : 0.8536 (+/- 0.0027)
f1        : 0.8036 (+/- 0.0048)
roc_auc   : 0.6472 (+/- 0.0178)

--- Evaluasi di Test Set ---
accuracy  

In [17]:
# ============================================================
# 8. RINGKASAN: CROSS-VALIDATION (TRAINING) & TEST SET
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# --- Ringkasan CV (training) ---
print(f"\n{'='*50}\nRINGKASAN CROSS-VALIDATION (Training Set)\n{'='*50}")
cv_summary_rows = []
for name, metrics in cv_results.items():
    row = {"classifier": name}
    for metric, (mean_val, std_val) in metrics.items():
        row[f"{metric}_mean"] = round(mean_val, 4)
        row[f"{metric}_std"] = round(std_val, 4)
    cv_summary_rows.append(row)

cv_summary_df = pd.DataFrame(cv_summary_rows)
print(cv_summary_df.to_string(index=False))

# --- Ringkasan Test Set ---
print(f"\n{'='*50}\nRINGKASAN EVALUASI TEST SET\n{'='*50}")
test_summary_rows = []
for name, metrics in test_results.items():
    row = {"classifier": name}
    for metric, val in metrics.items():
        row[metric] = round(val, 4)
    test_summary_rows.append(row)

test_summary_df = pd.DataFrame(test_summary_rows)
print(test_summary_df.to_string(index=False))


RINGKASAN CROSS-VALIDATION (Training Set)
        classifier  accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std  roc_auc_mean  roc_auc_std
LogisticRegression         0.8563        0.0026          0.8346         0.0073       0.8563      0.0026   0.8144  0.0055        0.7647       0.0109
               SVM         0.8536        0.0027          0.8354         0.0110       0.8536      0.0027   0.8036  0.0048        0.6472       0.0178
           XGBoost         0.8403        0.0063          0.8048         0.0110       0.8403      0.0063   0.8121  0.0080        0.7503       0.0100

RINGKASAN EVALUASI TEST SET
        classifier  accuracy  precision  recall     f1  roc_auc
LogisticRegression    0.8610     0.8425  0.8610 0.8202   0.7662
               SVM    0.8570     0.8370  0.8570 0.8093   0.6407
           XGBoost    0.8491     0.8170  0.8491 0.8215   0.7652


# MRMD+NN EMBEDDING

In [23]:
# ============================================================
# 7. TRAINING + CROSS VALIDATION (dari data TRAINING)
# ============================================================

cv = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_weighted",
    "roc_auc": "roc_auc",
}

cv_results = {}
test_results = {}   # <-- tambahan: simpan hasil evaluasi test set

for name, clf in classifiers.items():
    print(f"\n{'='*50}\nModel: {name}\n{'='*50}")

    # ---- Cross-validation di data training ----
    scores = cross_validate(clf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_results[name] = {
        metric: (np.mean(scores[f"test_{metric}"]), np.std(scores[f"test_{metric}"]))
        for metric in scoring
    }

    print("--- Cross-Validation (Training Set) ---")
    for metric, (mean_val, std_val) in cv_results[name].items():
        print(f"{metric:10s}: {mean_val:.4f} (+/- {std_val:.4f})")

    # ---- Fit ke seluruh training set, evaluasi ke test set ----
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]  # asumsi biner

    test_metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted"),
        "recall": recall_score(y_test, y_pred, average="weighted"),
        "f1": f1_score(y_test, y_pred, average="weighted"),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
    test_results[name] = test_metrics   # <-- simpan untuk ringkasan nanti

    print("\n--- Evaluasi di Test Set ---")
    for metric, val in test_metrics.items():
        print(f"{metric:10s}: {val:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


Model: LogisticRegression
--- Cross-Validation (Training Set) ---
accuracy  : 0.8662 (+/- 0.0050)
precision : 0.8485 (+/- 0.0092)
recall    : 0.8662 (+/- 0.0050)
f1        : 0.8374 (+/- 0.0082)
roc_auc   : 0.8068 (+/- 0.0143)

--- Evaluasi di Test Set ---
accuracy  : 0.8667
precision : 0.8469
recall    : 0.8667
f1        : 0.8390
roc_auc   : 0.8047

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.98      0.93      3135
           1       0.67      0.24      0.36       564

    accuracy                           0.87      3699
   macro avg       0.78      0.61      0.64      3699
weighted avg       0.85      0.87      0.84      3699

Confusion Matrix:
[[3069   66]
 [ 427  137]]

Model: SVM
--- Cross-Validation (Training Set) ---
accuracy  : 0.8663 (+/- 0.0043)
precision : 0.8482 (+/- 0.0077)
recall    : 0.8663 (+/- 0.0043)
f1        : 0.8389 (+/- 0.0058)
roc_auc   : 0.7691 (+/- 0.0227)

--- Evaluasi di Test Set ---
accuracy  

In [24]:
# ============================================================
# 8. RINGKASAN: CROSS-VALIDATION (TRAINING) & TEST SET
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# --- Ringkasan CV (training) ---
print(f"\n{'='*50}\nRINGKASAN CROSS-VALIDATION (Training Set)\n{'='*50}")
cv_summary_rows = []
for name, metrics in cv_results.items():
    row = {"classifier": name}
    for metric, (mean_val, std_val) in metrics.items():
        row[f"{metric}_mean"] = round(mean_val, 4)
        row[f"{metric}_std"] = round(std_val, 4)
    cv_summary_rows.append(row)

cv_summary_df = pd.DataFrame(cv_summary_rows)
print(cv_summary_df.to_string(index=False))

# --- Ringkasan Test Set ---
print(f"\n{'='*50}\nRINGKASAN EVALUASI TEST SET\n{'='*50}")
test_summary_rows = []
for name, metrics in test_results.items():
    row = {"classifier": name}
    for metric, val in metrics.items():
        row[metric] = round(val, 4)
    test_summary_rows.append(row)

test_summary_df = pd.DataFrame(test_summary_rows)
print(test_summary_df.to_string(index=False))


RINGKASAN CROSS-VALIDATION (Training Set)
        classifier  accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std  roc_auc_mean  roc_auc_std
LogisticRegression         0.8662        0.0050          0.8485         0.0092       0.8662      0.0050   0.8374  0.0082        0.8068       0.0143
               SVM         0.8663        0.0043          0.8482         0.0077       0.8663      0.0043   0.8389  0.0058        0.7691       0.0227
           XGBoost         0.8633        0.0042          0.8456         0.0052       0.8633      0.0042   0.8490  0.0044        0.8289       0.0054

RINGKASAN EVALUASI TEST SET
        classifier  accuracy  precision  recall     f1  roc_auc
LogisticRegression    0.8667     0.8469  0.8667 0.8390   0.8047
               SVM    0.8729     0.8583  0.8729 0.8466   0.7510
           XGBoost    0.8659     0.8475  0.8659 0.8506   0.8359


# MEAN MODUS

In [40]:
# ============================================================
# 7. TRAINING + CROSS VALIDATION (dari data TRAINING)
# ============================================================

cv = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",   # default average="binary"
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

cv_results = {}
test_results = {}   # <-- tambahan: simpan hasil evaluasi test set

for name, clf in classifiers.items():
    print(f"\n{'='*50}\nModel: {name}\n{'='*50}")

    # ---- Cross-validation di data training ----
    scores = cross_validate(clf, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_results[name] = {
        metric: (np.mean(scores[f"test_{metric}"]), np.std(scores[f"test_{metric}"]))
        for metric in scoring
    }

    print("--- Cross-Validation (Training Set) ---")
    for metric, (mean_val, std_val) in cv_results[name].items():
        print(f"{metric:10s}: {mean_val:.4f} (+/- {std_val:.4f})")

    # ---- Fit ke seluruh training set, evaluasi ke test set ----
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]  # asumsi biner

    test_metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="binary"),
        "recall": recall_score(y_test, y_pred, average="binary"),
        "f1": f1_score(y_test, y_pred, average="binary"),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
    test_results[name] = test_metrics   # <-- simpan untuk ringkasan nanti

    print("\n--- Evaluasi di Test Set ---")
    for metric, val in test_metrics.items():
        print(f"{metric:10s}: {val:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


Model: LogisticRegression
--- Cross-Validation (Training Set) ---
accuracy  : 0.8050 (+/- 0.0035)
precision : 0.6904 (+/- 0.0229)
recall    : 0.3546 (+/- 0.0155)
f1        : 0.4681 (+/- 0.0116)
roc_auc   : 0.8137 (+/- 0.0070)

--- Evaluasi di Test Set ---
accuracy  : 0.8111
precision : 0.6934
recall    : 0.3682
f1        : 0.4810
roc_auc   : 0.8132

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.95      0.88      7447
           1       0.69      0.37      0.48      2322

    accuracy                           0.81      9769
   macro avg       0.76      0.66      0.68      9769
weighted avg       0.80      0.81      0.79      9769

Confusion Matrix:
[[7069  378]
 [1467  855]]

Model: SVM
--- Cross-Validation (Training Set) ---
accuracy  : 0.8206 (+/- 0.0056)
precision : 0.7591 (+/- 0.0264)
recall    : 0.3803 (+/- 0.0144)
f1        : 0.5066 (+/- 0.0160)
roc_auc   : 0.8426 (+/- 0.0032)

--- Evaluasi di Test Set ---
accuracy  

In [41]:
# ============================================================
# 8. RINGKASAN: CROSS-VALIDATION (TRAINING) & TEST SET
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# --- Ringkasan CV (training) ---
print(f"\n{'='*50}\nRINGKASAN CROSS-VALIDATION (Training Set)\n{'='*50}")
cv_summary_rows = []
for name, metrics in cv_results.items():
    row = {"classifier": name}
    for metric, (mean_val, std_val) in metrics.items():
        row[f"{metric}_mean"] = round(mean_val, 4)
        row[f"{metric}_std"] = round(std_val, 4)
    cv_summary_rows.append(row)

cv_summary_df = pd.DataFrame(cv_summary_rows)
print(cv_summary_df.to_string(index=False))

# --- Ringkasan Test Set ---
print(f"\n{'='*50}\nRINGKASAN EVALUASI TEST SET\n{'='*50}")
test_summary_rows = []
for name, metrics in test_results.items():
    row = {"classifier": name}
    for metric, val in metrics.items():
        row[metric] = round(val, 4)
    test_summary_rows.append(row)

test_summary_df = pd.DataFrame(test_summary_rows)
print(test_summary_df.to_string(index=False))


RINGKASAN CROSS-VALIDATION (Training Set)
        classifier  accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std  roc_auc_mean  roc_auc_std
LogisticRegression         0.8050        0.0035          0.6904         0.0229       0.3546      0.0155   0.4681  0.0116        0.8137       0.0070
               SVM         0.8206        0.0056          0.7591         0.0264       0.3803      0.0144   0.5066  0.0160        0.8426       0.0032
           XGBoost         0.8444        0.0024          0.7247         0.0077       0.5766      0.0032   0.6422  0.0045        0.8911       0.0044

RINGKASAN EVALUASI TEST SET
        classifier  accuracy  precision  recall     f1  roc_auc
LogisticRegression    0.8111     0.6934  0.3682 0.4810   0.8132
               SVM    0.8255     0.7506  0.3979 0.5201   0.8439
           XGBoost    0.8513     0.7315  0.5913 0.6540   0.9004
